# Testing libraries

In [ ]:
!pip3 install pyswisseph

In [ ]:
import swisseph as swe

swe.set_sid_mode(swe.SIDM_LAHIRI)
jd = swe.julday(2016, 6, 15)
coords = (41.39858935189999, 2.1853450576007365, 18)
coords = (1000, 10000, 0)
swe.set_topo(*coords) # ESMUC
xx, ret = swe.calc_ut(jd, swe.MOON)
xx # longitude, latitude, distance (AU), Speed in longitude (deg/day), Speed in latitude (deg/day), Speed in distance (AU/day)

In [ ]:
!pip3 install ephem


https://rhodesmill.org/pyephem/quick
https://rhodesmill.org/pyephem/radec
https://rhodesmill.org/pyephem/tutorial.html

In [1]:
def pretty(d, indent=0):
   for key, value in d.items():
      print('\t' * indent + str(key))
      if isinstance(value, dict):
         pretty(value, indent+1)
      else:
         print('\t' * (indent+1) + str(value))

In [9]:
import ephem
from zoneinfo import ZoneInfo

planets = [
    ephem.Moon(),
    # ephem.Mercury(),
    # ephem.Venus(),
    # ephem.Mars(),
    # ephem.Jupiter(),
    # ephem.Saturn(),
    # ephem.Uranus()
]

valsTemplate = {'ra': '', 'dec': '', 'mag': '', 'size': '', 'separation': ''}

measurings = dict.fromkeys((planet.name for planet in planets), {})


observer = ephem.Observer()
observer.name = "ESMUC"
observer.lat = '41.39858935189999'
observer.lon = '2.1853450576007365'
observer.elevation = 18  # meters
observer.pressure = 0


d = ephem.Date('2016/06/17 00:00')
zone = ZoneInfo('Europe/Madrid')
observer.date = d - 2*ephem.hour # Account for timezone (?)

rangDies = 1
stepHores = 1
resolucio = 22
resolucio = 24 - resolucio
print(int(rangDies * (24/resolucio)))

for i in range(int(rangDies * (24/resolucio))):
    for j, planet in enumerate(planets):
        planet.compute(observer)
        dateValues = dict({str(observer.date):{'ra': planet.ra, 'dec': planet.dec, 'mag': planet.mag, 'size': planet.size, 'separation': ephem.separation(observer, planet), 'alt': planet.alt, 'az': planet.az}})
        measurings[planet.name] =  {**measurings[planet.name], **dateValues}

        # print('%s measured from %s:' % (planet.name, observer.name))
        # print('%s, %s. Magnitude: %s, Size: %s' % (planet.ra, planet.dec, planet.mag, planet.size)) # Right Ascension, Declination (Apparent Topocentric Position, aka des del punt d'observació), magnitude
        # print(ephem.separation(observer, planet))
    observer.date += stepHores * ephem.hour

# pretty((measurings))


12


In [ ]:
import matplotlib.pylab as plt
import numpy as np

xAxis = np.linspace(2016, 2026, int(rangDies * (24/resolucio))) # TODO: NO FORÇAR EL RANG
planetsArray = {}

for j, planet in enumerate(planets):
    values = {'ra': {}, 'dec': {}, 'mag': {}, 'size': {}, 'separation': {}}

    for i in range(len(measurings[planet.name])):
        values["ra"][i] = list(measurings[planet.name].values())[i]["ra"]
        values["dec"][i] = list(measurings[planet.name].values())[i]["dec"]
        values["mag"][i] = list(measurings[planet.name].values())[i]["mag"]
        values["size"][i] = list(measurings[planet.name].values())[i]["size"]
        values["separation"][i] = list(measurings[planet.name].values())[i]["separation"]

    # pretty(values)

    planetsArray[planet.name] = values

    ax = plt.subplot(4, 2, j+1)
    
    ax.plot(xAxis, values["ra"].values(), 'b', label="ra")
    ax.plot(xAxis, values["dec"].values(), 'g', label="dec")
    ax.plot(xAxis, values["mag"].values(), 'y', label="mag")
    # ax.plot(xAxis, values["size"].values(), 'r', label="size")
    ax.plot(xAxis, values["separation"].values(), 'c', label="separation")
    ax.set_title(planet.name, fontsize=8)


plt.tight_layout()
plt.legend(loc=(1.1,0))
plt.show()

print(type(planetsArray))

# Get planets within viewing scope

Good tool to check if calculations are correct: https://stellarium-web.org/, https://heavens-above.com/

In [ ]:
# Using ephem.separation

coords = (str(observer.lat), str(observer.lon))
coords = ephem.Equatorial(*coords)

viewing_angle = ephem.degrees("180:00:00.0") # String = degrees
print(viewing_angle)

visiblePlanets = {}


for j in range(len(measurings["Moon"])):
    visiblePlanetsAtThisTime = {}
    for i, planet in enumerate(planets):
        thisPlanet = list(measurings[planet.name].values())[j]
        separation = thisPlanet['separation']
        if(separation < viewing_angle):
            time = list(measurings[planet.name].keys())[j]
            # print('%s is visible from %s at %s with %s degrees separation' % (planet.name, observer.name, time, separation))
            visiblePlanetsAtThisTime[planet.name] = {'ra' : thisPlanet['ra'], 'dec': thisPlanet['dec'], 'separation': separation, 'mag': planet.mag, 'size': planet.size}
    # print(visiblePlanetsAtThisTime)
    visiblePlanets[time] = {**visiblePlanetsAtThisTime}

# pretty(visiblePlanets)
pretty((visiblePlanets))


In [ ]:
# Using azalt

coords = (str(observer.lat), str(observer.lon))
coords = ephem.Equatorial(*coords)

viewing_angle = ephem.degrees("20:00:00.0") # String = degrees

visiblePlanets = {}

for j in range(len(measurings["Moon"])):
    visiblePlanetsAtThisTime = {}
    for i, planet in enumerate(planets):
        thisPlanet = list(measurings[planet.name].values())[j]
        az = thisPlanet['az']
        alt = thisPlanet['alt']
        if(alt > viewing_angle):
            time = list(measurings[planet.name].keys())[j]
            # print('%s is visible from %s at %s with %s degrees separation' % (planet.name, observer.name, time, separation))
            visiblePlanetsAtThisTime[planet.name] = {'ra' : thisPlanet['ra'], 'dec': thisPlanet['dec'], 'separation': separation, 'mag': planet.mag, 'size': planet.size, 'az': az, 'alt': alt}
    # print(visiblePlanetsAtThisTime)
    visiblePlanets[time] = {**visiblePlanetsAtThisTime}

# pretty(visiblePlanets)
pretty((visiblePlanets))


20:00:00.0
2016/6/16 22:00:00
	Moon
		ra
			14:59:21.45
		dec
			-13:14:59.1
		separation
			130:18:05.7
		mag
			-11.64
		size
			1755.510986328125
		az
			195:15:25.5
		alt
			34:03:31.8
2016/6/16 23:00:00
	Moon
		ra
			15:00:39.96
		dec
			-13:21:19.7
		separation
			130:18:05.7
		mag
			-11.64
		size
			1755.510986328125
		az
			211:17:11.5
		alt
			29:35:02.7
2016/6/17 00:00:00


# Fent servir MIDI

In [ ]:
import rtmidi
import time as t

midiout = rtmidi.MidiOut()
available_ports = midiout.get_ports()

if available_ports:
    midiout.open_port(0)
else:
    midiout.open_virtual_port("Virtual Output")

print(midiout.get_port_name(0))

with midiout:
    for i, date in enumerate(visiblePlanets):
        planetsVisibleNow = visiblePlanets[date]
        # print(date, planetsVisibleNow)
        notes_on = []
        notes_off = []
        for i, planet in enumerate(planetsVisibleNow):
            planetInfo = planetsVisibleNow[planet]
            note = planetInfo['separation'] * 127
            amplitude = abs(planetInfo['mag'] * 10)
            
            for j, plan in enumerate(planets):
                if planet == plan.name:
                    chan = j
                    break
                else:
                    chan = -1

            NOTE_ON = 0x90
            NOTE_OFF = 0x80
            status_on = NOTE_ON | (chan)
            status_off = NOTE_OFF | (chan)

            note_on = [status_on, note, amplitude]
            notes_on.append(note_on)
            note_off = [status_off, note, amplitude]
            notes_off.append(note_off)

        # print(notes_on, notes_off)
        for message in notes_on:
            midiout.send_message(message)
        t.sleep(.5)
        for message in notes_off:
            midiout.send_message(message)

# Fent servir OSC

In [ ]:
!pip3 install python-osc

In [ ]:
from pythonosc import udp_client

client = udp_client.SimpleUDPClient("127.0.0.1", 5050, timeout=10)



for i, date in enumerate(visiblePlanets):
        planetsVisibleNow = visiblePlanets[date]

        outMessages = []

        for i, planet in enumerate(planetsVisibleNow):
            planetInfo = planetsVisibleNow[planet]
            print(planetInfo)
            
            for j, plan in enumerate(planets):
                if planet == plan.name:
                    chan = j
                    break
                else:
                    chan = -1

            message = (
                "/"+planet, [
                    planetInfo['ra'],
                    planetInfo['dec'],
                    planetInfo['separation'],
                    planetInfo['mag'],
                    planetInfo['size']
                ]
            )
            outMessages.append(message)

        # print(notes_on, notes_off)
        for message in outMessages:
            print('Sending message', message)
            client.send_message(*message)
        t.sleep(.2)


In [3]:
import ephem
from zoneinfo import ZoneInfo

observer = ephem.Observer()
observer.name = "Observation Point"
observer.lat = '41.39858935189999'
observer.lon = '2.1853450576007365'
observer.elevation = 18  # meters
observer.pressure = 0

d = ephem.Date('2016/06/17 00:00')
zone = ZoneInfo("Europe/Madrid")
observer.date = d - 2*ephem.hour # Account for timezone (?)

for _ in range(24):
    moon = ephem.Moon(observer)
    print('%s, az: %s alt: %s' % (ephem.date(observer.date + 2*ephem.hour), moon.az, moon.alt))
    observer.date += ephem.hour

2016/6/17 00:00:00, az: 195:15:25.5 alt: 34:03:31.8
2016/6/17 01:00:00, az: 211:17:11.5 alt: 29:35:02.7
2016/6/17 02:00:00, az: 225:14:04.3 alt: 22:39:57.1
2016/6/17 03:00:00, az: 237:11:35.0 alt: 14:01:20.4
2016/6/17 04:00:00, az: 247:39:16.9 alt: 4:14:48.0
2016/6/17 05:00:00, az: 257:12:44.1 alt: -6:13:14.5
2016/6/17 06:00:00, az: 266:28:32.6 alt: -17:02:50.8
2016/6/17 07:00:00, az: 276:07:35.3 alt: -27:56:50.5
2016/6/17 08:00:00, az: 287:04:21.1 alt: -38:36:14.4
2016/6/17 09:00:00, az: 300:43:43.3 alt: -48:32:48.9
2016/6/17 10:00:00, az: 319:19:51.6 alt: -56:54:34.2
2016/6/17 11:00:00, az: 345:03:43.4 alt: -62:05:11.4
2016/6/17 12:00:00, az: 15:06:48.5 alt: -62:10:44.3
2016/6/17 13:00:00, az: 41:03:48.1 alt: -57:09:22.3
2016/6/17 14:00:00, az: 59:52:41.1 alt: -48:54:30.6
2016/6/17 15:00:00, az: 73:41:17.7 alt: -39:04:13.6
2016/6/17 16:00:00, az: 84:45:15.3 alt: -28:31:50.2
2016/6/17 17:00:00, az: 94:30:40.2 alt: -17:46:22.8
2016/6/17 18:00:00, az: 103:52:30.5 alt: -7:07:28.7
2016/6/